In [1]:
!pip install requests beautifulsoup4

In [7]:
# 巨潮网 科创板 招股说明书爬虫
# 功能：仅下载 50 家 → 首次公开发行股票并在科创板上市招股说明书
# Jupyter 直接运行

import requests
import time
import random
import os
from tqdm import tqdm
from bs4 import BeautifulSoup

# ===================== 配置 =====================
SAVE_DIR = "科创板招股书"
os.makedirs(SAVE_DIR, exist_ok=True)

BASE_URL = "https://www.cninfo.com.cn/new/hisAnnouncement/query"
PDF_BASE = "https://static.cninfo.com.cn/"

MAX_DOWNLOAD = 50  # 只下载50家

UA_LIST = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/128.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
]

HEADERS = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Referer": "https://www.cninfo.com.cn/",
    "X-Requested-With": "XMLHttpRequest"
}

# ===================== 获取单页公告 =====================
def get_announcement_page(page_num):
    headers = HEADERS.copy()
    headers["User-Agent"] = random.choice(UA_LIST)

    data = {
        "pageNum": page_num,
        "pageSize": 50,
        "tabName": "fulltext",
        "column": "sse",
        "plate": "kcb",
        "stock": "",
        "searchkey": "招股说明书",
        "category": "",
        "seDate": "2019-07-22~2026-12-31",
        "sortName": "time",
        "sortType": "desc",
        "isHLtitle": "false"
    }

    try:
        resp = requests.post(BASE_URL, headers=headers, data=data, timeout=20)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"第{page_num}页请求失败：{e}")
        return None

# ===================== 下载PDF =====================
def download_pdf(pdf_url, sec_code, sec_name):
    filename = f"{sec_code}_{sec_name}_科创板招股说明书.pdf"
    filename = filename.replace("/", "").replace("\\", "")
    save_path = os.path.join(SAVE_DIR, filename)

    if os.path.exists(save_path):
        print(f"✅ 已存在：{filename}")
        return True

    try:
        resp = requests.get(pdf_url, stream=True, timeout=30)
        resp.raise_for_status()
        total_size = int(resp.headers.get("content-length", 0))

        with open(save_path, "wb") as f, tqdm(total=total_size, unit="B", unit_scale=True, desc=sec_code) as bar:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))

        print(f"✅ 下载成功：{filename}")
        return True

    except Exception as e:
        print(f"❌ 下载失败：{e}")
        return False

# ===================== 主程序 =====================
def crawl_kcb_50():
    print("🚀 开始爬取巨潮网科创板招股说明书（仅下载50家）\n")
    page = 1
    count = 0

    while count < MAX_DOWNLOAD:
        print(f"\n===== 正在爬取第 {page} 页 =====")
        data = get_announcement_page(page)

        if not data or not data.get("announcements"):
            print("无更多数据")
            break

        for ann in data["announcements"]:
            title = ann["announcementTitle"]
            sec_code = ann["secCode"]
            sec_name = ann["secName"]
            url = ann.get("adjunctUrl")

            # 严格匹配：只抓科创板正式招股书（排除摘要、申报稿等）
            if "首次公开发行股票并在科创板上市" in title and "招股说明书" in title:
                if any(x in title for x in ["摘要", "申报", "修正", "问询", "提示"]):
                    continue

                if count >= MAX_DOWNLOAD:
                    break

                if url:
                    pdf_url = PDF_BASE + url
                    success = download_pdf(pdf_url, sec_code, sec_name)
                    if success:
                        count += 1
                        print(f"📊 已下载 {count}/{MAX_DOWNLOAD}\n")
                        time.sleep(random.uniform(2, 4))

        page += 1
        time.sleep(1)

    print(f"\n🎉 任务完成！共下载 {count} 家科创板招股说明书！")

# ===================== Jupyter 直接运行 =====================
crawl_kcb_50()

🚀 开始爬取巨潮网科创板招股说明书（仅下载50家）


===== 正在爬取第 1 页 =====


688808: 100%|█████████████████████████████████████████████████████████████████████| 8.88M/8.88M [00:00<00:00, 42.2MB/s]


✅ 下载成功：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 1/50



688820: 100%|█████████████████████████████████████████████████████████████████████| 7.75M/7.75M [00:00<00:00, 41.3MB/s]


✅ 下载成功：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 2/50



...

✅ 下载成功：688781_视涯科技_科创板招股说明书.pdf
📊 已下载 5/50


===== 正在爬取第 2 页 =====
✅ 已存在：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 6/50

✅ 已存在：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 7/50

✅ 已存在：688811_有研复材_科创板招股说明书.pdf
📊 已下载 8/50

✅ 已存在：688813_泰金新能_科创板招股说明书.pdf
📊 已下载 9/50

✅ 已存在：688781_视涯科技_科创板招股说明书.pdf
📊 已下载 10/50


===== 正在爬取第 3 页 =====
✅ 已存在：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 11/50

✅ 已存在：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 12/50

✅ 已存在：688811_有研复材_科创板招股说明书.pdf
📊 已下载 13/50



688809: 100%|█████████████████████████████████████████████████████████████████████| 5.61M/5.61M [00:00<00:00, 41.6MB/s]


✅ 下载成功：688809_强一股份_科创板招股说明书.pdf
📊 已下载 14/50



688805: 100%|█████████████████████████████████████████████████████████████████████| 5.56M/5.56M [00:00<00:00, 33.2MB/s]


✅ 下载成功：688805_健信超导_科创板招股说明书.pdf
📊 已下载 15/50


===== 正在爬取第 4 页 =====
✅ 已存在：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 16/50

✅ 已存在：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 17/50

✅ 已存在：688811_有研复材_科创板招股说明书.pdf
📊 已下载 18/50

✅ 已存在：688809_强一股份_科创板招股说明书.pdf
📊 已下载 19/50

✅ 已存在：688805_健信超导_科创板招股说明书.pdf
📊 已下载 20/50


===== 正在爬取第 5 页 =====
✅ 已存在：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 21/50

✅ 已存在：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 22/50

✅ 已存在：688811_有研复材_科创板招股说明书.pdf
📊 已下载 23/50

✅ 已存在：688809_强一股份_科创板招股说明书.pdf
📊 已下载 24/50

✅ 已存在：688805_健信超导_科创板招股说明书.pdf
📊 已下载 25/50


===== 正在爬取第 6 页 =====
✅ 已存在：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 26/50

✅ 已存在：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 27/50

✅ 已存在：688811_有研复材_科创板招股说明书.pdf
📊 已下载 28/50

✅ 已存在：688809_强一股份_科创板招股说明书.pdf
📊 已下载 29/50

✅ 已存在：688805_健信超导_科创板招股说明书.pdf
📊 已下载 30/50


===== 正在爬取第 7 页 =====
✅ 已存在：688808_联讯仪器_科创板招股说明书.pdf
📊 已下载 31/50

✅ 已存在：688820_盛合晶微_科创板招股说明书.pdf
📊 已下载 32/50

✅ 已存在：688811_有研复材_科创板招股说明书.pdf
📊 已下载 33/50

✅ 已存在：688813_泰金新能_科创板招股说明书.pdf
📊 已下载 34/50

✅ 已存在：688781_视涯科技_科创板招股说明书.